# 👩‍💻 Feature Selection and Extraction for Housing Price Prediction
## 📋 Overview
In this lab, you'll tackle a real-world machine learning challenge: reducing the number of features in the Ames Housing Dataset while maintaining or improving predictive performance. You'll implement feature selection using Recursive Feature Elimination (RFE) and dimensionality reduction using Principal Component Analysis (PCA), then compare their effectiveness for a linear regression model. These techniques are essential for any data scientist working with high-dimensional datasets as they help improve model efficiency, reduce overfitting, and increase interpretability.
## 🎯 Learning Outcomes
By the end of this lab, you will be able to:

- Apply Recursive Feature Elimination to identify the most important features in a dataset
- Implement Principal Component Analysis for dimensionality reduction
- Compare and evaluate the performance of models using different feature selection techniques
- Make informed decisions about feature selection trade-offs in real-world scenarios

## 🚀 Starting Point
Access the starter code provided below. You'll need a Python environment with the following libraries:

- pandas
- numpy
- scikit-learn
- matplotlib (optional, for visualization)

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Load the Ames Housing dataset
ames = fetch_openml(name="house_prices", as_frame=True)
data = ames.data
data['SalePrice'] = ames.target

## Task 1: Explore the Dataset
**Context:** Before applying any feature selection technique, it's important to understand the dataset you're working with. Real estate analysts often start by exploring housing data to get a sense of the available features and their distributions.

**Steps:**

1. Examine the first few rows of the dataset using the `head()` method
2. Get statistical summaries of numerical features using `describe()`
3. Check for missing values in the dataset using methods like `isna().sum()`
4. Handle missing values in numerical features using appropriate strategies

In [2]:
# Explore the dataset
# YOUR CODE HERE
print("Shape of Dataset: ", data.shape)

print("-------- Top Five Rows --------")
display(data.head())

print("-------- Dataset Information --------")
print(data.info())

print("-------- Summary --------")
display(data.describe().T)

# Handle missing values
# YOUR CODE HERE
# Handle missing values
data = data.select_dtypes(include=[np.number])  # Select only numerical features for simplicity
data = data.fillna(data.median())


# Define features and target
# YOUR CODE HERE
X = data.drop('SalePrice', axis=1)
y = data['SalePrice'].astype(float)

Shape of Dataset:  (1460, 81)
-------- Top Five Rows --------


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1.0,60.0,RL,65.0,8450.0,Pave,None,Reg,Lvl,AllPub,...,0.0,None,None,None,0.0,2.0,2008.0,WD,Normal,208500.0
1,2.0,20.0,RL,80.0,9600.0,Pave,None,Reg,Lvl,AllPub,...,0.0,None,None,None,0.0,5.0,2007.0,WD,Normal,181500.0
2,3.0,60.0,RL,68.0,11250.0,Pave,None,IR1,Lvl,AllPub,...,0.0,None,None,None,0.0,9.0,2008.0,WD,Normal,223500.0
3,4.0,70.0,RL,60.0,9550.0,Pave,None,IR1,Lvl,AllPub,...,0.0,None,None,None,0.0,2.0,2006.0,WD,Abnorml,140000.0
4,5.0,60.0,RL,84.0,14260.0,Pave,None,IR1,Lvl,AllPub,...,0.0,None,None,None,0.0,12.0,2008.0,WD,Normal,250000.0


-------- Dataset Information --------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   float64
 1   MSSubClass     1460 non-null   float64
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   float64
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual   

,count,mean,std,min,25%,50%,75%,max
Id,1460.0,730.500000,421.610009,1.0,365.75,730.5,1095.25,1460.0
MSSubClass,1460.0,56.897260,42.300571,20.0,20.00,50.0,70.00,190.0
LotFrontage,1201.0,70.049958,24.284752,21.0,59.00,69.0,80.00,313.0
LotArea,1460.0,10516.828082,9981.264932,1300.0,7553.50,9478.5,11601.50,215245.0
OverallQual,1460.0,6.099315,1.382997,1.0,5.00,6.0,7.00,10.0
OverallCond,1460.0,5.575342,1.112799,1.0,5.00,5.0,6.00,9.0
YearBuilt,1460.0,1971.267808,30.202904,1872.0,1954.00,1973.0,2000.00,2010.0
YearRemodAdd,1460.0,1984.865753,20.645407,1950.0,1967.00,1994.0,2004.00,2010.0
MasVnrArea,1452.0,103.685262,181.066207,0.0,0.00,0.0,166.00,1600.0
BsmtFinSF1,1460.0,443.639726,456.098091,0.0,0.00,383.5,712.25,5644.0


**💡 Tip:** For simplicity in this lab, consider using only numerical features and handling missing values with median imputation.

**⚙️ Test Your Work:**

- Print the shape of your processed dataset
- Verify that there are no missing values in the data you'll use for modeling
- Expected output: A confirmation of the dataset dimensions and features you'll be working with

## Task 2: Apply Recursive Feature Elimination (RFE)
**Context:** Real estate companies often want to know which housing attributes are most predictive of sales price. RFE helps identify the most important features while eliminating redundant or less important ones.

**Steps:**

1. Create a Linear Regression model to use as the base estimator
2. Initialize the RFE with the model, specifying to select the top 5 features
3. Fit RFE to your data
4. Extract and display the selected features

In [3]:
# Apply Recursive Feature Elimination
# YOUR CODE HERE

# Apply Recursive Feature Elimination (RFE)
model = LinearRegression()
rfe = RFE(model, n_features_to_select=5)
X_rfe = rfe.fit_transform(X, y)

# Extract selected features
# YOUR CODE HERE

selected_features = X.columns[rfe.support_]
print("Selected Features:", selected_features)

Selected Features: Index(['OverallQual', 'BsmtFullBath', 'FullBath', 'Fireplaces', 'GarageCars'], dtype='object')


**💡 Tip:** The `RFE` class has a `support_` attribute that shows which features were selected. You can use this with your original feature names to identify the selected features.

**⚙️ Test Your Work:**

- Print the names of the selected features
- Expected output: A list of the 5 most important features for predicting house prices

## Task 3: Implement Principal Component Analysis (PCA)
**Context:** In many real-world datasets including real estate data, features may be correlated. PCA transforms the original features into uncorrelated principal components that capture the maximum variance in the data.

**Steps:**

1. Standardize the features using `StandardScaler`
2. Initialize PCA with 2 components to start
3. Fit and transform the data using PCA
4. Examine the explained variance ratio to understand how much information is retained

In [4]:
# Standardize the features
# YOUR CODE HERE


# Explore PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
# YOUR CODE HERE
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Examine explained variance
# YOUR CODE HERE
print("Explained Variance Ratio by PCA:", pca.explained_variance_ratio_)

Explained Variance Ratio by PCA: [0.19253349 0.0865939 ]


**💡 Tip:** Always standardize your data before applying PCA since it is sensitive to the scale of the features.
    
**⚙️ Test Your Work:**

- Print the explained variance ratio of the principal components
- Expected output: The percentage of variance explained by each principal component

## Task 4: Evaluate Model Performance
**Context:** Data scientists must compare different approaches to determine which yields the best model. Here, you'll evaluate whether feature selection with RFE or dimensionality reduction with PCA results in better predictive performance.

**Steps:**

1. Split the RFE-selected data and the PCA-transformed data into training and testing sets
2. Train a Linear Regression model on each training set
3. Make predictions on the test sets
4. Calculate and compare performance metrics (R-squared and MSE) for both approaches

In [5]:
# Evaluate RFE model
# YOUR CODE HERE

# Evaluate Model Performance with RFE features
X_train, X_test, y_train, y_test = train_test_split(X_rfe, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
y_pred_rfe = model.predict(X_test)

rfe_r2= r2_score(y_test, y_pred_rfe)
rfe_mse= mean_squared_error(y_test, y_pred_rfe)

print("RFE Model - R-squared:", rfe_r2)
print("RFE Model - MSE:", rfe_mse)


# Evaluate PCA model
# YOUR CODE HERE

# Evaluate Model Performance with PCA-transformed data
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2, random_state=42)
model.fit(X_train_pca, y_train_pca)
y_pred_pca = model.predict(X_test_pca)

pca_r2= r2_score(y_test_pca, y_pred_pca)
pca_mse=  mean_squared_error(y_test_pca, y_pred_pca)

print("PCA Model - R-squared:", pca_r2)
print("PCA Model - MSE:", pca_mse)

# Compare performance
# YOUR CODE HERE

print("\n-- Performance Compare --")
if rfe_r2 > pca_r2:
    print("✅ The RFE-based model explains more variance and has better predictive performance.")
    print("RFE selects the most informative features directly, which can preserve interpretability and relevance.")
elif pca_r2 > rfe_r2:
    print("✅ The PCA-based model performed better in terms of variance explained.")
    print("PCA transforms features into orthogonal components, which may help when features are highly correlated.")
else:
    print("⚖️ Both models achieved similar R² scores. Consider other metrics or domain needs to choose between them.")

if rfe_mse < pca_mse:
    print("📉 RFE also had lower MSE, indicating more accurate predictions on average.")
elif pca_mse < rfe_mse:
    print("📉 PCA had lower MSE, suggesting tighter prediction errors.")
else:
    print("📉 Both models had similar MSE values.")

RFE Model - R-squared: 0.7128562035020778
RFE Model - MSE: 2202486587.50922
PCA Model - R-squared: 0.7810237737235992
PCA Model - MSE: 1679619087.158815

-- Performance Compare --
✅ The PCA-based model performed better in terms of variance explained.
PCA transforms features into orthogonal components, which may help when features are highly correlated.
📉 PCA had lower MSE, suggesting tighter prediction errors.


**💡 Tip:** Use the same random state when splitting data to ensure a fair comparison between models.
    
**⚙️ Test Your Work:**

- Print the R-squared and MSE values for both models
- Expected output: Performance metrics showing how well each model predicts house prices

## Task 5: Analyze and Document Findings
**Context:** In a real-world scenario, you would need to communicate your findings to stakeholders. This involves analyzing the trade-offs between different approaches and making recommendations.

**Steps:**

1. Compare the performance of the RFE and PCA approaches
2. Discuss the interpretability advantage of RFE (knowing specific important features) versus the potential information preservation of PCA
3. Document which features RFE selected and why they might be important for house price prediction

In [6]:
# Document your findings
# YOUR CODE HERE

# Analyze and Document Findings
print("\nAnalysis of Results:")
print(f"The top 5 features selected by RFE are: {', '.join(selected_features)}")
print(f"RFE model performance: R² = {r2_score(y_test, y_pred_rfe):.4f}, MSE = {mean_squared_error(y_test, y_pred_rfe):.2f}")
print(f"PCA model performance: R² = {r2_score(y_test_pca, y_pred_pca):.4f}, MSE = {mean_squared_error(y_test_pca, y_pred_pca):.2f}")


Analysis of Results:
The top 5 features selected by RFE are: OverallQual, BsmtFullBath, FullBath, Fireplaces, GarageCars
RFE model performance: R² = 0.7129, MSE = 2202486587.51
PCA model performance: R² = 0.7810, MSE = 1679619087.16


**💡 Tip:** Consider both quantitative metrics and qualitative aspects like interpretability in your analysis.
    
**⚙️ Test Your Work:**

- Write a concise summary of your findings
- Expected output: A clear analysis comparing the two approaches with specific metrics and insights

## ✅ Success Checklist
- Successfully loaded and preprocessed the Ames Housing dataset
- Applied RFE to identify the 5 most important features
- Implemented PCA for dimensionality reduction
- Trained and evaluated linear regression models using both approaches
- Compared performance metrics between RFE and PCA approaches
- Documented insights about feature importance and selection trade-offs
- Code runs without errors

## 🔍 Common Issues & Solutions
**Problem:** RFE takes a long time to run. 

**Solution:** Start with a smaller subset of features or use `RFECV` with cross-validation to find the optimal number of features more efficiently.

**Problem:** Poor model performance even after feature selection. 

**Solution:** Consider trying different base estimators for RFE or exploring other preprocessing techniques for the dataset.

**Problem:** PCA components are difficult to interpret. 

**Solution:** This is a natural trade-off with PCA. If interpretability is critical, feature selection methods like RFE might be more appropriate than PCA.

## 🔑 Key Points
- Feature selection techniques like RFE help identify the most predictive features, improving model interpretability.
- PCA reduces dimensionality while preserving variance but sacrifices the direct interpretability of features.
- The choice between feature selection and dimensionality reduction depends on your specific goals and requirements.
- Always evaluate and compare model performance to make data-driven decisions about feature engineering.

## 💻 Exemplar Solution

<details>

<summary><strong>Click HERE to see an exemplar solution</strong></summary>    
    
```python
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


# Load the Ames Housing dataset
ames = fetch_openml(name="house_prices", as_frame=True)
data = ames.data
data['SalePrice'] = ames.target
print(data.head())
print(data.describe())


# Basic data preprocessing
# Handle missing values
data = data.select_dtypes(include=[np.number])  # Select only numerical features for simplicity
data = data.fillna(data.median())


# Define features and target
X = data.drop('SalePrice', axis=1)
y = data['SalePrice'].astype(float)


# Apply Recursive Feature Elimination (RFE)
model = LinearRegression()
rfe = RFE(model, n_features_to_select=5)
X_rfe = rfe.fit_transform(X, y)


selected_features = X.columns[rfe.support_]
print("Selected Features:", selected_features)


# Explore PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)


print("Explained Variance Ratio by PCA:", pca.explained_variance_ratio_)


# Evaluate Model Performance with RFE features
X_train, X_test, y_train, y_test = train_test_split(X_rfe, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
y_pred_rfe = model.predict(X_test)


print("RFE Model - R-squared:", r2_score(y_test, y_pred_rfe))
print("RFE Model - MSE:", mean_squared_error(y_test, y_pred_rfe))


# Evaluate Model Performance with PCA-transformed data
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2, random_state=42)
model.fit(X_train_pca, y_train_pca)
y_pred_pca = model.predict(X_test_pca)


print("PCA Model - R-squared:", r2_score(y_test_pca, y_pred_pca))
print("PCA Model - MSE:", mean_squared_error(y_test_pca, y_pred_pca))


# Analyze and Document Findings
print("\nAnalysis of Results:")
print(f"The top 5 features selected by RFE are: {', '.join(selected_features)}")
print(f"RFE model performance: R² = {r2_score(y_test, y_pred_rfe):.4f}, MSE = {mean_squared_error(y_test, y_pred_rfe):.2f}")
print(f"PCA model performance: R² = {r2_score(y_test_pca, y_pred_pca):.4f}, MSE = {mean_squared_error(y_test_pca, y_pred_pca):.2f}")

```    